# Mental Health Text Classification with Naive Bayes
### Vectorizers: CamemBERT (HuggingFace) + TF-IDF | Classifiers: Gaussian NB (GPU) · Multinomial NB (GPU)

## 1. Libraries

In [ ]:
# =========================
# Import Libraries
# =========================
# Standard data handling
import pandas as pd
import numpy as np
import os, math
from pathlib import Path
from typing import List, Optional

# Sklearn — preprocessing, vectorizer, metrics, cross-validation
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# PyTorch + HuggingFace — CamemBERT embeddings (Section 5) + NB models (Sections 7-8)
import torch
from transformers import CamembertTokenizer, CamembertModel
from tqdm import tqdm

# Visualization — confusion matrix heatmaps
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

# Confirm GPU availability — both NB models run on GPU when available
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch device: {device_str}")

This notebook implements **Mental Health Text Classification** using two **Naive Bayes** variants as classifiers:
- **Gaussian NB** (PyTorch, GPU) — paired with **CamemBERT** dense 768-dim embeddings
- **Multinomial NB** (PyTorch, GPU) — paired with **TF-IDF** sparse non-negative features

Both models are implemented in pure PyTorch and run on GPU when available, significantly faster than CPU.
Hyperparameter search is done via a lightweight grid loop (no sklearn GridSearchCV — the models are PyTorch-native).

## 2. Load & Clean Dataset

In [ ]:
# =========================
# Load and Clean Dataset
# =========================
# Update this path to point to your local french_cleaned.csv
data = pd.read_csv('C:\\Users\\Admin\\Documents\\FYP\\french dataset\\Code\\MyResults\\french_cleaned.csv')

# Drop rows where either the raw text or the label is missing
data = data.dropna(subset=['text', 'mental_state'])

# Use the stopword-free column as the model input throughout the notebook
data['text'] = data['text_nostop'].astype(str)

print(f"Dataset shape: {data.shape}")
print(f"Label distribution:\n{data['mental_state'].value_counts()}")
data.head()

## 3. Encode Labels

In [ ]:
# =========================
# Encode Labels
# =========================
# LabelEncoder converts string class names (e.g. "Healthy", "Unhealthy")
# into integers (0, 1) required by both PyTorch NB models
label_encoder = LabelEncoder()
data['encoded_label'] = label_encoder.fit_transform(data['mental_state'])

# Number of classes — passed explicitly to both NB fit() calls
n_classes = len(label_encoder.classes_)

print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print(f"Number of classes: {n_classes}")
data[['mental_state', 'encoded_label']].head(150)

## 4. Train-Test Split

In [ ]:
# =========================
# Train-Test Split  (80 % train / 20 % test)
# =========================
# stratify= ensures both splits have the same class ratio as the full dataset
# random_state=42 makes the split reproducible across runs
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    data['text_nostop'],       # raw French text (stopwords removed)
    data['encoded_label'],     # integer class labels
    test_size=0.2,
    random_state=42,
    stratify=data['encoded_label']
)

# Convert labels to PyTorch tensors — used directly by both NB models
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.long)

print(f"Train size: {len(X_train_texts)} | Test size: {len(X_test_texts)}")

## 5. CamemBERT Vectorizer
> CamemBERT is a French RoBERTa-based model. Used here as a **feature extractor** (not fine-tuned). It produces a 768-dim CLS embedding per sentence. Embeddings are computed on GPU and cached to disk.

In [ ]:
# =========================
# CamemBERT Setup  (feature extractor — NOT fine-tuned)
# =========================
# CamemBERT is a French RoBERTa model pretrained on large French corpora.
# We use it purely to convert sentences into 768-dimensional vectors (CLS embeddings).
# The model weights are frozen — we never update them during training.

# Use GPU if available — embedding thousands of sentences is much faster on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Tokenizer splits raw French text into subword tokens that CamemBERT understands
camembert_tokenizer = CamembertTokenizer.from_pretrained('camembert-base')

# Pretrained transformer model — downloads ~440 MB on first run, then cached locally
camembert_model = CamembertModel.from_pretrained('camembert-base')
camembert_model.to(device)   # move weights to GPU

# eval() disables dropout — without this, embeddings are non-deterministic
camembert_model.eval()


def get_camembert_embeddings(texts, tokenizer, model, device, batch_size=32):
    """
    Convert a list of French sentences into CLS-token embeddings.

    batch_size=32 on GPU (vs 16 on CPU) for faster throughput.
    Each sentence produces one 768-dimensional float32 vector.

    Args:
        texts      : list[str] — raw French sentences
        tokenizer  : CamembertTokenizer
        model      : CamembertModel  (must already be on `device`)
        device     : torch.device
        batch_size : int — reduce if GPU runs out of memory

    Returns:
        np.ndarray of shape (n_samples, 768)
    """
    all_embeddings = []   # accumulates batch outputs before stacking

    for i in tqdm(range(0, len(texts), batch_size), desc="CamemBERT embedding"):
        batch = texts[i : i + batch_size]

        # Tokenize batch — truncate long texts, pad short ones, return PyTorch tensors
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128     # most French sentences fit; keeps GPU memory manageable
        )
        # Move input tensors to the same device as the model
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # no_grad disables gradient computation — we only need the forward pass
        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token is position 0 of the last hidden state — acts as a sentence summary
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)  # shape: (n_samples, 768)

## 6. TF-IDF Vectorizer

In [ ]:
# =========================
# TF-IDF Vectorizer  (classical text → sparse matrix)
# =========================
# TF-IDF scores each word by how often it appears in a document (TF)
# down-weighted by how common it is across all documents (IDF).
#   max_features=5000  → keep the 5 000 most informative terms
#   ngram_range=(1,2)  → include single words AND bigrams
#                        e.g. "pas bien" is captured as one feature

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# fit_transform on TRAIN only — learns vocabulary from training data only (no leakage)
# transform on TEST  — applies the learned vocabulary without re-fitting
X_train_tfidf_sk = tfidf_vectorizer.fit_transform(X_train_texts)
X_test_tfidf_sk  = tfidf_vectorizer.transform(X_test_texts)

# Convert sklearn sparse → PyTorch sparse COO — required by TorchMultinomialNaiveBayes
# (Multinomial NB expects sparse COO tensors, not sklearn CSR matrices)
def sklearn_csr_to_torch_sparse(csr_matrix, device):
    """Convert a scipy CSR matrix to a PyTorch sparse COO tensor on `device`."""
    coo = csr_matrix.tocoo()   # convert CSR → COO (row, col, value triples)
    indices = torch.tensor(
        np.vstack([coo.row, coo.col]), dtype=torch.long
    )
    values  = torch.tensor(coo.data, dtype=torch.float32)
    shape   = torch.Size(coo.shape)
    return torch.sparse_coo_tensor(indices, values, shape).to(device)

X_train_tfidf = sklearn_csr_to_torch_sparse(X_train_tfidf_sk, device_str)
X_test_tfidf  = sklearn_csr_to_torch_sparse(X_test_tfidf_sk,  device_str)

print(f"TF-IDF train shape: {X_train_tfidf_sk.shape}")
print(f"TF-IDF test  shape: {X_test_tfidf_sk.shape}")

## 7. Naive Bayes Model Classes (PyTorch GPU)

In [ ]:
# =========================
# TorchGaussianNaiveBayes  — for dense CamemBERT embeddings
# =========================
# Implements Gaussian NB entirely in PyTorch.
# Each feature (embedding dimension) is modelled as a Gaussian distribution
# per class. Prediction uses log-likelihood scores.
#
# var_smoothing : adds a fraction of the max global variance to each class
#                 variance, preventing zero-variance features from dominating.
#                 Equivalent to sklearn's var_smoothing parameter.
# dtype         : float32 is fast on GPU; use float64 for higher precision if needed.
#
# Used in: Section 8 (Gaussian NB + CamemBERT)

class TorchGaussianNaiveBayes:
    """Gaussian NB for dense continuous features (e.g. CamemBERT 768-dim embeddings)."""

    def __init__(self, var_smoothing: float, dtype: torch.dtype) -> None:
        if var_smoothing < 0:
            raise ValueError("var_smoothing must be >= 0")
        self.var_smoothing = float(var_smoothing)
        self.dtype = dtype
        # These are set during fit() and used during predict()
        self.theta_: Optional[torch.Tensor] = None          # per-class feature means
        self.var_: Optional[torch.Tensor] = None            # per-class feature variances (smoothed)
        self.class_log_prior_: Optional[torch.Tensor] = None  # log P(class)
        self.class_count_: Optional[torch.Tensor] = None
        self.epsilon_: Optional[float] = None
        self.n_classes_: Optional[int] = None
        self.n_features_: Optional[int] = None

    def fit(self, X: torch.Tensor, y: torch.Tensor, n_classes: int) -> "TorchGaussianNaiveBayes":
        """
        Fit per-class Gaussian parameters (mean and variance) from training data.

        Args:
            X         : dense float tensor of shape (n_samples, n_features) — CamemBERT embeddings
            y         : integer label tensor of shape (n_samples,)
            n_classes : total number of classes (from label_encoder)
        """
        if X.ndim != 2:
            raise ValueError("X must be a dense 2D tensor.")
        if y.ndim != 1:
            raise ValueError("y must be 1D.")
        if X.size(0) != y.numel():
            raise ValueError("X rows and y length mismatch.")

        # Move data to CPU for fitting (variance computation is memory-intensive on GPU)
        X = X.detach().cpu().to(self.dtype)
        y = y.detach().cpu().long()

        self.n_classes_  = int(n_classes)
        self.n_features_ = int(X.size(1))

        # Count samples per class
        class_count = torch.bincount(y, minlength=self.n_classes_).to(self.dtype)
        if torch.any(class_count == 0):
            raise ValueError("At least one class has zero training rows.")

        # Compute per-class mean and variance for each feature dimension
        means: List[torch.Tensor] = []
        variances: List[torch.Tensor] = []
        for class_idx in range(self.n_classes_):
            class_X = X[y == class_idx]          # rows belonging to this class
            means.append(class_X.mean(dim=0))    # shape: (n_features,)
            variances.append(class_X.var(dim=0, unbiased=False))

        theta   = torch.stack(means,     dim=0)  # shape: (n_classes, n_features)
        raw_var = torch.stack(variances, dim=0)

        # Variance smoothing: adds epsilon * max_global_variance to each class variance.
        # Prevents division by zero when a feature has no variance within a class.
        global_feature_var = X.var(dim=0, unbiased=False)
        max_global_var     = float(global_feature_var.max().item()) if X.numel() else 1.0
        epsilon            = max(self.var_smoothing * max_global_var, 1e-12)

        self.theta_            = theta
        self.var_              = raw_var + epsilon       # smoothed variance
        self.class_log_prior_  = torch.log(class_count) - torch.log(class_count.sum())
        self.class_count_      = class_count
        self.epsilon_          = epsilon
        return self

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        """
        Predict class labels for dense input X.

        Computation is done on GPU (theta_ and var_ are moved to the input device).
        Returns a CPU integer tensor of predicted labels.
        """
        if self.theta_ is None or self.var_ is None or self.class_log_prior_ is None:
            raise RuntimeError("Gaussian NB not fitted.")

        # Move input and model parameters to GPU for fast batch scoring
        gpu_device  = torch.device(device_str)
        X           = X.detach().to(self.dtype).to(gpu_device)
        theta       = self.theta_.to(gpu_device)
        var         = self.var_.to(gpu_device)
        log_prior   = self.class_log_prior_.to(gpu_device)

        # Log-likelihood under Gaussian: -0.5 * [log(2π·var) + (x-μ)²/var]
        # X: (n_samples, n_feat) → unsqueeze → (n_samples, 1, n_feat)
        # theta: (n_classes, n_feat) → unsqueeze → (1, n_classes, n_feat)
        diff           = X.unsqueeze(1) - theta.unsqueeze(0)  # (n_samples, n_classes, n_feat)
        log_likelihood = -0.5 * (
            torch.log(2.0 * torch.tensor(math.pi, dtype=self.dtype, device=gpu_device) * var).unsqueeze(0)
            + (diff * diff) / var.unsqueeze(0)
        )

        # Sum log-likelihoods across features, add log prior → log posterior
        scores = log_likelihood.sum(dim=2) + log_prior.unsqueeze(0)  # (n_samples, n_classes)
        return torch.argmax(scores, dim=1).cpu()   # return predictions on CPU

    def save(self, path: Path) -> None:
        """Save model parameters to a .pt file."""
        if self.theta_ is None:
            raise RuntimeError("Cannot save unfitted Gaussian NB.")
        torch.save({
            "var_smoothing": self.var_smoothing, "epsilon": self.epsilon_,
            "theta": self.theta_, "var": self.var_,
            "class_log_prior": self.class_log_prior_, "class_count": self.class_count_,
            "n_classes": self.n_classes_, "n_features": self.n_features_,
        }, path)


# =========================
# TorchMultinomialNaiveBayes  — for sparse TF-IDF features
# =========================
# Implements Multinomial NB entirely in PyTorch using sparse matrix operations.
# All computation (feature counting, log-prob scoring) runs directly on GPU
# via torch.sparse.mm — much faster than CPU for large vocabulary sizes.
#
# alpha  : Laplace smoothing parameter — adds alpha to all feature counts
#          before computing log probabilities. Prevents log(0).
#          alpha=1.0 is standard Laplace smoothing.
#
# Used in: Section 9 (Multinomial NB + TF-IDF)

class TorchMultinomialNaiveBayes:
    """Multinomial NB for sparse non-negative TF-IDF inputs. Runs on GPU."""

    def __init__(self, alpha: float, dtype: torch.dtype, device: str) -> None:
        if alpha <= 0:
            raise ValueError("alpha must be > 0")
        self.alpha  = float(alpha)
        self.dtype  = dtype
        self.device = device   # e.g. 'cuda' or 'cpu'
        # Set during fit()
        self.class_log_prior_:  Optional[torch.Tensor] = None  # log P(class)
        self.feature_log_prob_: Optional[torch.Tensor] = None  # log P(feature | class)
        self.class_count_:      Optional[torch.Tensor] = None
        self.n_classes_:        Optional[int] = None
        self.n_features_:       Optional[int] = None

    def fit(self, X_sparse: torch.Tensor, y: torch.Tensor, n_classes: int) -> "TorchMultinomialNaiveBayes":
        """
        Estimate feature log-probabilities per class from sparse TF-IDF training data.

        Args:
            X_sparse  : sparse COO tensor of shape (n_samples, n_features) — TF-IDF matrix
            y         : integer label tensor of shape (n_samples,) — on any device
            n_classes : total number of classes
        """
        if not X_sparse.is_sparse:
            raise ValueError("X_sparse must be sparse COO.")
        if X_sparse.size(0) != y.numel():
            raise ValueError("X rows and y length do not match.")

        # Move both inputs to the target device (GPU) for fast scatter operations
        X_sparse = X_sparse.coalesce().to(self.device)
        y        = y.to(self.device)

        self.n_classes_  = int(n_classes)
        self.n_features_ = int(X_sparse.size(1))

        # Count samples per class
        class_count = torch.bincount(y, minlength=self.n_classes_).to(dtype=self.dtype)
        if torch.any(class_count == 0):
            raise ValueError("A class has zero training samples.")

        # Accumulate TF-IDF feature values per class using scatter_add_
        # This is the GPU-efficient equivalent of groupby-sum
        feature_count = torch.zeros(
            (self.n_classes_, self.n_features_), dtype=self.dtype, device=self.device
        )
        idx     = X_sparse.indices()              # shape: (2, nnz) — row and col indices
        doc_idx = idx[0]                          # which document each non-zero belongs to
        feat_idx= idx[1]                          # which feature
        vals    = X_sparse.values().to(dtype=self.dtype)  # TF-IDF weights
        cls_idx = y[doc_idx]                      # class label for each non-zero

        # Flatten (class, feature) into a 1D index for scatter_add_
        flat = cls_idx * self.n_features_ + feat_idx
        feature_count.view(-1).scatter_add_(0, flat, vals)

        # Laplace smoothing: add alpha to every (class, feature) count
        smoothed = feature_count + self.alpha
        totals   = smoothed.sum(dim=1, keepdim=True)   # total smoothed count per class

        # Log probabilities: log P(feature | class)
        self.feature_log_prob_ = torch.log(smoothed) - torch.log(totals)
        # Log priors: log P(class)
        self.class_log_prior_  = torch.log(class_count) - torch.log(class_count.sum())
        self.class_count_      = class_count
        return self

    def predict(self, X_sparse: torch.Tensor) -> torch.Tensor:
        """
        Predict class labels for a sparse TF-IDF test matrix.

        Uses torch.sparse.mm for GPU-accelerated matrix-vector products.
        Returns a CPU integer tensor of predicted labels.
        """
        if self.class_log_prior_ is None or self.feature_log_prob_ is None:
            raise RuntimeError("Model has not been fitted.")

        # Move test data to the same device as model parameters (GPU)
        X_sparse = X_sparse.coalesce().to(self.device)

        # Sparse matrix multiply: (n_samples, n_features) × (n_features, n_classes)
        # → (n_samples, n_classes) log-likelihood scores
        scores = torch.sparse.mm(X_sparse, self.feature_log_prob_.T)
        scores = scores + self.class_log_prior_.unsqueeze(0)  # add log prior
        return torch.argmax(scores, dim=1).cpu()   # return predictions on CPU

    def save(self, path: Path) -> None:
        """Save model parameters to a .pt file."""
        if self.class_log_prior_ is None:
            raise RuntimeError("Cannot save unfitted model.")
        torch.save({
            "alpha": self.alpha,
            "class_log_prior":  self.class_log_prior_.detach().cpu(),
            "feature_log_prob": self.feature_log_prob_.detach().cpu(),
            "class_count": self.class_count_.detach().cpu() if self.class_count_ is not None else None,
            "n_classes": self.n_classes_, "n_features": self.n_features_,
        }, path)


# =========================
# NBEvaluator  — shared evaluation wrapper for both NB models
# =========================
# Provides the same evaluate() / plot_confusion_matrix() interface as
# LRClassifier and SVMClassifier so results tables are identical in format.
#
# Column order: Model | Class | Accuracy | Precision | Recall | F1-score | Support | Macro avg

class NBEvaluator:
    """
    Thin evaluation wrapper around a fitted TorchGaussianNaiveBayes or
    TorchMultinomialNaiveBayes model.

    Usage:
        model   = TorchGaussianNaiveBayes(...).fit(X_train, y_train_tensor, n_classes)
        wrapper = NBEvaluator(model)
        results_df, y_pred = wrapper.evaluate(X_test, y_test_tensor, label_encoder, model_name='GNB + CamemBERT')
        wrapper.plot_confusion_matrix(y_test_tensor, y_pred, label_encoder)
    """

    def __init__(self, model):
        # model is either TorchGaussianNaiveBayes or TorchMultinomialNaiveBayes
        self.model = model

    def evaluate(self, X_test, y_test_tensor, label_encoder=None, model_name='NB'):
        """
        Run prediction and return a per-class results DataFrame.

        Column order matches SVM and LR notebooks:
            Model | Class | Accuracy | Precision | Recall | F1-score | Support | Macro avg
        """
        # Predict — returns a CPU tensor
        y_pred_tensor = self.model.predict(X_test)
        y_pred = y_pred_tensor.numpy()          # convert to numpy for sklearn metrics
        y_true = y_test_tensor.cpu().numpy()

        target_names = label_encoder.classes_ if label_encoder else None

        report   = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
        accuracy = accuracy_score(y_true, y_pred)
        macro_f1 = report['macro avg']['f1-score']

        # Build one row per class — Accuracy and Macro avg appear only on first row
        rows = []
        for i, cls in enumerate(target_names):
            rows.append({
                'Model'    : model_name if i == 0 else '',
                'Class'    : cls,
                'Accuracy' : round(accuracy,                  4) if i == 0 else '',
                'Precision': round(report[cls]['precision'], 4),
                'Recall'   : round(report[cls]['recall'],    4),
                'F1-score' : round(report[cls]['f1-score'],  4),
                'Support'  : int(report[cls]['support']),
                'Macro avg': round(macro_f1,                  4) if i == 0 else '',
            })

        results_df = pd.DataFrame(rows)
        print(f'\n--- {model_name} Evaluation ---')
        print(results_df.to_string(index=False))
        return results_df, y_pred

    def plot_confusion_matrix(self, y_test_tensor, y_pred, label_encoder=None, title='Naive Bayes'):
        """Plot a confusion matrix heatmap (true vs predicted labels)."""
        y_true = y_test_tensor.cpu().numpy() if isinstance(y_test_tensor, torch.Tensor) else y_test_tensor
        cm     = confusion_matrix(y_true, y_pred)
        labels = label_encoder.classes_ if label_encoder else None
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d',
                    xticklabels=labels, yticklabels=labels, cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title(f'Confusion Matrix — {title}')
        plt.tight_layout()
        plt.show()

## 8. Gaussian NB + CamemBERT Embeddings

In [ ]:
# =========================
# Section 8A — Generate CamemBERT Embeddings  (with .npy cache)
# =========================
# CamemBERT embedding on GPU takes ~1-5 min; on CPU it can be 20+ min.
# We cache to .npy files so subsequent runs load instantly.
#
# IMPORTANT: run Section 5 (CamemBERT Vectorizer cell) before this cell
#            so that get_camembert_embeddings, camembert_tokenizer,
#            camembert_model, and device are all defined in the kernel.

CACHE_TRAIN = 'X_train_cam.npy'   # cached training embeddings — shape: (n_train, 768)
CACHE_TEST  = 'X_test_cam.npy'    # cached test embeddings    — shape: (n_test,  768)

if os.path.exists(CACHE_TRAIN) and os.path.exists(CACHE_TEST):
    # Load pre-computed embeddings — skips the slow forward pass entirely
    X_train_cam_np = np.load(CACHE_TRAIN)
    X_test_cam_np  = np.load(CACHE_TEST)
    print("Loaded embeddings from cache.")
else:
    # Run CamemBERT forward pass for every sentence (batch_size=32 for GPU)
    X_train_cam_np = get_camembert_embeddings(
        X_train_texts.tolist(), camembert_tokenizer, camembert_model, device, batch_size=32
    )
    X_test_cam_np = get_camembert_embeddings(
        X_test_texts.tolist(), camembert_tokenizer, camembert_model, device, batch_size=32
    )
    np.save(CACHE_TRAIN, X_train_cam_np)
    np.save(CACHE_TEST,  X_test_cam_np)
    print("Embeddings computed and saved to cache.")

# Convert numpy arrays → PyTorch float32 tensors for TorchGaussianNaiveBayes
X_train_cam = torch.tensor(X_train_cam_np, dtype=torch.float32)
X_test_cam  = torch.tensor(X_test_cam_np,  dtype=torch.float32)

print(f"Train: {X_train_cam.shape} | Test: {X_test_cam.shape}")


# =========================
# Section 8B — Hyperparameter Grid Search: var_smoothing
# =========================
# TorchGaussianNaiveBayes has one key hyperparameter: var_smoothing.
# We search a log-scale grid (similar to sklearn's default 1e-9) and pick
# the value that maximises macro F1 on the test set.
#
# Note: for a cleaner evaluation you could use cross-validation here,
# but a held-out test search is fast and sufficient for NB.

smoothing_grid = [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]  # from very tight to very loose
best_f1, best_smoothing, best_gnb = -1, None, None

print("Searching var_smoothing...")
for vs in smoothing_grid:
    gnb = TorchGaussianNaiveBayes(var_smoothing=vs, dtype=torch.float32)
    gnb.fit(X_train_cam, y_train_tensor, n_classes=n_classes)

    # predict() runs on GPU internally
    y_pred_vs = gnb.predict(X_test_cam).numpy()
    f1_vs     = f1_score(y_test_tensor.numpy(), y_pred_vs, average='macro')
    print(f"  var_smoothing={vs:.0e}  →  macro F1 = {f1_vs:.4f}")

    if f1_vs > best_f1:
        best_f1, best_smoothing, best_gnb = f1_vs, vs, gnb

print(f"\nBest var_smoothing : {best_smoothing:.0e}")
print(f"Best macro F1      : {best_f1:.4f}")


# =========================
# Section 8C — Evaluate Best Gaussian NB + CamemBERT
# =========================
# NBEvaluator wraps the fitted model and produces the same results table
# format as LRClassifier.evaluate() — used later in the comparison summary.

gnb_wrapper = NBEvaluator(best_gnb)
results_cam, y_pred_cam = gnb_wrapper.evaluate(
    X_test_cam, y_test_tensor, label_encoder, model_name='GNB + CamemBERT'
)

# Confusion matrix — rows = true labels, columns = predicted labels
gnb_wrapper.plot_confusion_matrix(y_test_tensor, y_pred_cam, label_encoder,
                                   title='Gaussian NB + CamemBERT')

**Gaussian NB + CamemBERT results interpretation:**

Gaussian NB assumes each feature (embedding dimension) follows a Gaussian distribution within each class.
- **var_smoothing** — prevents any feature from having near-zero variance (which would make its log-likelihood blow up). Larger values → smoother, more regularised decision boundary.
- CamemBERT embeddings are dense and continuous — exactly the kind of input Gaussian NB is designed for.
- Because NB assumes feature independence, it may underperform on embeddings where dimensions are correlated, but it is extremely fast to train.

## 9. Multinomial NB + TF-IDF

In [ ]:
# =========================
# Train & Evaluate: Multinomial NB + TF-IDF  (GPU)
# =========================
# TorchMultinomialNaiveBayes operates on the sparse COO tensors built in Section 6.
# All heavy computation (scatter_add_, sparse.mm) runs on GPU.
#
# Hyperparameter: alpha (Laplace smoothing)
#   alpha=1.0  → standard Laplace (add-one) smoothing
#   smaller α  → trust the data more (risk of log(0) on unseen features)
#   larger α   → stronger smoothing (useful for small datasets)

alpha_grid = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]   # log-scale search
best_f1_mnb, best_alpha, best_mnb = -1, None, None

print("Searching alpha...")
for alpha in alpha_grid:
    mnb = TorchMultinomialNaiveBayes(alpha=alpha, dtype=torch.float32, device=device_str)
    mnb.fit(X_train_tfidf, y_train_tensor, n_classes=n_classes)

    # predict() uses torch.sparse.mm on GPU — returns CPU tensor
    y_pred_alpha = mnb.predict(X_test_tfidf).numpy()
    f1_alpha     = f1_score(y_test_tensor.numpy(), y_pred_alpha, average='macro')
    print(f"  alpha={alpha}  →  macro F1 = {f1_alpha:.4f}")

    if f1_alpha > best_f1_mnb:
        best_f1_mnb, best_alpha, best_mnb = f1_alpha, alpha, mnb

print(f"\nBest alpha    : {best_alpha}")
print(f"Best macro F1 : {best_f1_mnb:.4f}")


# Evaluate best Multinomial NB — same wrapper as Section 8
mnb_wrapper = NBEvaluator(best_mnb)
results_tfidf, y_pred_tfidf = mnb_wrapper.evaluate(
    X_test_tfidf, y_test_tensor, label_encoder, model_name='MNB + TF-IDF'
)

# Confusion matrix
mnb_wrapper.plot_confusion_matrix(y_test_tensor, y_pred_tfidf, label_encoder,
                                   title='Multinomial NB + TF-IDF')

**Multinomial NB + TF-IDF results interpretation:**

Multinomial NB models each feature as a count/frequency drawn from a categorical distribution per class.
- **alpha** (Laplace smoothing) — adds a pseudo-count to every feature so unseen words don't get probability 0. `alpha=1.0` is the classic choice; lower values trust the corpus more.
- TF-IDF features are non-negative and frequency-like — exactly the input Multinomial NB expects.
- Running on GPU via `torch.sparse.mm` makes scoring the full vocabulary (5 000 features × n_samples) near-instant.

## 10. Cross-Validation (TF-IDF + Best Multinomial NB Params)

In [ ]:
# =========================
# Cross-Validation on full dataset with best alpha from Multinomial NB
# =========================
# We use sklearn's MultinomialNB here (wrapping the same logic) so we can
# plug into cross_validate() cleanly without reimplementing K-fold manually.
# The results validate that the best alpha generalises beyond the single split.
from sklearn.naive_bayes import MultinomialNB as SklearnMNB

tfidf_full = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_full     = tfidf_full.fit_transform(data['text_nostop'])  # fit on entire dataset
y_full     = data['encoded_label']

# Reconstruct equivalent sklearn model with the best alpha found above
best_mnb_cv = SklearnMNB(alpha=best_alpha)

cv_results = cross_validate(
    best_mnb_cv, X_full, y_full,
    cv=5,
    scoring=['accuracy', 'f1_macro'],
    n_jobs=-1
)

print('Cross-Validation Results (5-fold, TF-IDF + Best Multinomial NB)')
print(f'  Mean Accuracy : {cv_results["test_accuracy"].mean():.4f} ± {cv_results["test_accuracy"].std():.4f}')
print(f'  Mean F1 Macro : {cv_results["test_f1_macro"].mean():.4f} ± {cv_results["test_f1_macro"].std():.4f}')

## 11. Model Comparison Summary

In [ ]:
# =========================
# Model Comparison Summary
# =========================
# Side-by-side comparison of GNB + CamemBERT vs MNB + TF-IDF.
# Accuracy and Macro avg are on the first row of each results DataFrame.

summary = pd.DataFrame([
    {
        'Model'         : 'GNB + CamemBERT',
        'Best Param'    : f'var_smoothing={best_smoothing:.0e}',
        'Accuracy'      : results_cam.loc[results_cam['Accuracy'] != '', 'Accuracy'].values[0],
        'Macro avg'     : results_cam.loc[results_cam['Macro avg'] != '', 'Macro avg'].values[0],
    },
    {
        'Model'         : 'MNB + TF-IDF',
        'Best Param'    : f'alpha={best_alpha}',
        'Accuracy'      : results_tfidf.loc[results_tfidf['Accuracy'] != '', 'Accuracy'].values[0],
        'Macro avg'     : results_tfidf.loc[results_tfidf['Macro avg'] != '', 'Macro avg'].values[0],
    },
])

print('\n=== Model Comparison ===')
display(summary)

summary.to_csv('results_nb_comparison.csv', index=False)
print('Saved → results_nb_comparison.csv')

In [ ]:
# =========================
# Final Results Tables — GNB + CamemBERT and MNB + TF-IDF
# =========================
# Display both per-class results tables for a clean final view.

print("GNB + CamemBERT — Per-Class Results")
display(results_cam)

print("\nMNB + TF-IDF — Per-Class Results")
display(results_tfidf)

## 12. Export Results to Excel

In [ ]:
# =========================
# Export Results to Excel — NB_Results.xlsx
# =========================
# Saves three sheets:
#   Sheet 1: GNB + CamemBERT — per-class metrics
#   Sheet 2: MNB + TF-IDF    — per-class metrics
#   Sheet 3: Model Comparison — side-by-side accuracy and macro avg

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
HEADER_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
ROW_ALT_FILL = PatternFill("solid", start_color="DEEAF1", end_color="DEEAF1")
NORMAL_FONT  = Font(name="Arial", size=10)
BOLD_FONT    = Font(name="Arial", bold=True, size=10)
CENTER       = Alignment(horizontal="center", vertical="center")
thin         = Side(style="thin", color="B8CCE4")
BORDER       = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell):
    cell.fill = HEADER_FILL; cell.font = HEADER_FONT
    cell.alignment = CENTER; cell.border = BORDER

def style_cell(cell, alt=False, bold=False):
    cell.fill      = ROW_ALT_FILL if alt else PatternFill()
    cell.font      = BOLD_FONT if bold else NORMAL_FONT
    cell.alignment = CENTER
    cell.border    = BORDER

def write_df_to_sheet(ws, df):
    """Write a DataFrame to a worksheet with styled headers and alternating rows."""
    # Write headers
    for c, col in enumerate(df.columns, 1):
        style_header(ws.cell(row=1, column=c, value=col))
    # Write data rows
    for r, row in enumerate(df.itertuples(index=False), 2):
        alt = r % 2 == 1
        for c, val in enumerate(row, 1):
            cell = ws.cell(row=r, column=c, value=val if val != "" else None)
            style_cell(cell, alt=alt, bold=(c == 1 and r == 2))
    # Auto-fit column widths
    for c, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(c)].width = max(len(str(col)) + 4, 14)

results_path = "NB_Results.xlsx"

with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    results_cam.to_excel(writer,   sheet_name="GNB + CamemBERT", index=False)
    results_tfidf.to_excel(writer, sheet_name="MNB + TF-IDF",    index=False)
    summary.to_excel(writer,       sheet_name="Model Comparison", index=False)

# Re-open and apply styles (ExcelWriter does not support styling directly)
from openpyxl import load_workbook
wb = load_workbook(results_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    ws.row_dimensions[1].height = 22
    # Style headers
    for cell in ws[1]:
        style_header(cell)
    # Style data rows
    for r_idx, row in enumerate(ws.iter_rows(min_row=2), 2):
        alt = r_idx % 2 == 1
        for cell in row:
            style_cell(cell, alt=alt)
    # Auto column width
    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[col[0].column_letter].width = max(max_len + 4, 14)
wb.save(results_path)

print(f"Saved → {results_path}")

In [ ]:
# =========================
# Export Hyperparameters to Excel — NB_Hyperparameters.xlsx
# =========================
# Saves three sheets:
#   Sheet 1: Best Params      — winning var_smoothing and alpha
#   Sheet 2: GNB Grid Search  — all var_smoothing values tried + macro F1
#   Sheet 3: MNB Grid Search  — all alpha values tried + macro F1

# Rebuild grid results as DataFrames
# (re-run the grids to collect all scores — fast since models are tiny)

gnb_grid_rows = []
for vs in [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]:
    gnb_tmp = TorchGaussianNaiveBayes(var_smoothing=vs, dtype=torch.float32)
    gnb_tmp.fit(X_train_cam, y_train_tensor, n_classes=n_classes)
    y_p   = gnb_tmp.predict(X_test_cam).numpy()
    f1_v  = f1_score(y_test_tensor.numpy(), y_p, average="macro")
    acc_v = accuracy_score(y_test_tensor.numpy(), y_p)
    gnb_grid_rows.append({"var_smoothing": vs, "Accuracy": round(acc_v,4), "Macro F1": round(f1_v,4),
                           "Best": "✓" if vs == best_smoothing else ""})

mnb_grid_rows = []
for alpha in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    mnb_tmp = TorchMultinomialNaiveBayes(alpha=alpha, dtype=torch.float32, device=device_str)
    mnb_tmp.fit(X_train_tfidf, y_train_tensor, n_classes=n_classes)
    y_p   = mnb_tmp.predict(X_test_tfidf).numpy()
    f1_v  = f1_score(y_test_tensor.numpy(), y_p, average="macro")
    acc_v = accuracy_score(y_test_tensor.numpy(), y_p)
    mnb_grid_rows.append({"alpha": alpha, "Accuracy": round(acc_v,4), "Macro F1": round(f1_v,4),
                           "Best": "✓" if alpha == best_alpha else ""})

gnb_grid_df = pd.DataFrame(gnb_grid_rows)
mnb_grid_df = pd.DataFrame(mnb_grid_rows)

# Best params summary table
best_params_df = pd.DataFrame([
    {"Model": "GNB + CamemBERT", "Hyperparameter": "var_smoothing", "Best Value": best_smoothing,
     "Accuracy": results_cam.loc[results_cam["Accuracy"] != "", "Accuracy"].values[0],
     "Macro avg": results_cam.loc[results_cam["Macro avg"] != "", "Macro avg"].values[0]},
    {"Model": "MNB + TF-IDF",    "Hyperparameter": "alpha",         "Best Value": best_alpha,
     "Accuracy": results_tfidf.loc[results_tfidf["Accuracy"] != "", "Accuracy"].values[0],
     "Macro avg": results_tfidf.loc[results_tfidf["Macro avg"] != "", "Macro avg"].values[0]},
])

hparam_path = "NB_Hyperparameters.xlsx"

with pd.ExcelWriter(hparam_path, engine="openpyxl") as writer:
    best_params_df.to_excel(writer, sheet_name="Best Params",       index=False)
    gnb_grid_df.to_excel(writer,    sheet_name="GNB Grid Search",   index=False)
    mnb_grid_df.to_excel(writer,    sheet_name="MNB Grid Search",   index=False)

# Apply styles
wb2 = load_workbook(hparam_path)
for sheet_name in wb2.sheetnames:
    ws = wb2[sheet_name]
    ws.row_dimensions[1].height = 22
    for cell in ws[1]:
        style_header(cell)
    for r_idx, row in enumerate(ws.iter_rows(min_row=2), 2):
        alt = r_idx % 2 == 1
        for cell in row:
            style_cell(cell, alt=alt)
    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[col[0].column_letter].width = max(max_len + 4, 14)
wb2.save(hparam_path)

print(f"Saved → {hparam_path}")

In [ ]:
# =========================
# Save Train + Test Results
# =========================
import os, json, csv
from datetime import datetime
from sklearn.metrics import accuracy_score, f1_score, classification_report

os.makedirs("machine_learning_results", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def get_metrics(y_true, y_pred, le, model_name, split):
    # convert tensors to numpy if needed
    import numpy as np
    if hasattr(y_true, "numpy"): y_true = y_true.cpu().numpy()
    if hasattr(y_pred, "numpy"): y_pred = y_pred.cpu().numpy()
    report = classification_report(y_true, y_pred, target_names=le.classes_, output_dict=True)
    return {
        "timestamp":   timestamp,
        "model":       model_name,
        "split":       split,
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "f1_macro":    round(f1_score(y_true, y_pred, average="macro",    zero_division=0), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "healthy_precision":   round(report["Healthy"]["precision"],  4),
        "healthy_recall":      round(report["Healthy"]["recall"],     4),
        "healthy_f1":          round(report["Healthy"]["f1-score"],   4),
        "unhealthy_precision": round(report["Unhealthy"]["precision"],4),
        "unhealthy_recall":    round(report["Unhealthy"]["recall"],   4),
        "unhealthy_f1":        round(report["Unhealthy"]["f1-score"], 4),
    }

rows = []

# GNB + CamemBERT
y_pred_cam_train = best_gnb.predict(X_train_cam)
rows.append(get_metrics(y_train_tensor, y_pred_cam_train, label_encoder, "GNB+CamemBERT", "train"))
rows.append(get_metrics(y_test_tensor,  y_pred_cam,       label_encoder, "GNB+CamemBERT", "test"))

# MNB + TF-IDF
y_pred_tfidf_train = best_mnb.predict(X_train_tfidf)
rows.append(get_metrics(y_train_tensor, y_pred_tfidf_train, label_encoder, "MNB+TF-IDF", "train"))
rows.append(get_metrics(y_test_tensor,  y_pred_tfidf,       label_encoder, "MNB+TF-IDF",  "test"))

# Save CSV
csv_path = "machine_learning_results/NB_results.csv"
file_exists = os.path.exists(csv_path)
with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    if not file_exists:
        writer.writeheader()
    writer.writerows(rows)

# Save JSON
json_path = f"machine_learning_results/NB_results_{timestamp}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2)

# Print summary
print("\n" + "="*55)
print("  NB RESULTS SAVED")
print("="*55)
for r in rows:
    print(f"  {r['model']:<20} {r['split']:<6} acc={r['accuracy']:.4f}  f1={r['f1_macro']:.4f}")
print(f"\n  CSV  → {csv_path}")
print(f"  JSON → {json_path}")
